# Supervised Fine-Tuning (SFT) on GEAP

This notebook is a **thin demo**: every step calls into the tested
`geap_tuning` package rather than re-implementing logic. See
[`docs/notes/tuning-apis.md`](../docs/notes/tuning-apis.md) for the API details.

SFT trains the base model to reproduce the `model` turn of each `contents`
record. Here the task is customer-support **intent classification** (labels:
`billing`, `technical`, `account`, `shipping`, `other`).

> **Requires live GCP and incurs tuning cost.** Have a real `.env` and
> `gcloud auth` in place before running the tune/eval cells.

In [ ]:
from geap_tuning.config import genai_client, load_config

cfg = load_config()
client = genai_client(cfg)
cfg

## 1. Build the dataset

Deterministic train/val/test splits from a tiny hand-written source set.

In [ ]:
from geap_tuning.sft.data import build_sft_dataset

paths = build_sft_dataset("../datasets/sft_support_intent")
paths

In [ ]:
import json
from pathlib import Path

first = Path(paths["train"]).read_text(encoding="utf-8").splitlines()[0]
print(json.dumps(json.loads(first), indent=2))

## 2. Stage the splits to GCS

In [ ]:
from geap_tuning.gcs import upload_file

train_uri = upload_file(paths["train"], f"{cfg.bucket}/sft_support_intent/train.jsonl")
val_uri = upload_file(paths["val"], f"{cfg.bucket}/sft_support_intent/val.jsonl")
train_uri, val_uri

## 3. Launch the tuning job and wait

Reuse an existing job with the same display name if one exists (cost control).

In [ ]:
from geap_tuning.jobs import find_tuning_job_by_display_name, wait_for_tuning_job
from geap_tuning.sft.tune import launch_sft_job

DISPLAY_NAME = "geap-sft-support-intent"

job = find_tuning_job_by_display_name(client, DISPLAY_NAME)
if job is None:
    job = launch_sft_job(client, train_uri=train_uri, val_uri=val_uri, display_name=DISPLAY_NAME)
job = wait_for_tuning_job(client, job.name)
job.state

## 4. Evaluate the tuned endpoint

In [ ]:
from geap_tuning.inference import generate
from geap_tuning.jobs import tuned_endpoint
from geap_tuning.sft.data import SUPPORT_TICKETS, build_records, split_dataset
from geap_tuning.sft.evaluate import run_eval

endpoint = tuned_endpoint(job)
_, _, test_pairs = split_dataset(SUPPORT_TICKETS)
metrics = run_eval(
    build_records(test_pairs),
    predict_fn=lambda user_text: generate(client, endpoint, user_text),
)
print(f"Test accuracy: {metrics['accuracy']:.3f}")

## Next steps

- **Preference tuning (DPO)** — same `tunings.tune` call with
  `method="PREFERENCE_TUNING"` + `beta` and a `completions`/`score` dataset
  (`notebooks/02_preference_tuning.ipynb`, future).
- **RLFT** — a `references` dataset plus a reward function over REST `v1beta1`
  (`notebooks/03_rlft.ipynb`, future).